In [ ]:
# Evaluation Metrics
# - Supervised (Classification + Regression)

In [ ]:
# Small dataset = K-fold or Startified K-fold cross validation
# Time-series forecasting = Time-based split, train on past observations and validate on future observations

In [ ]:
# Classification Metrics

In [16]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.20,
    random_state = 42,
    stratify = y
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Training accuracy:", model.score(X_train, y_train))
print("Testing accuracy:", accuracy_score(y_test, y_pred))

Training accuracy: 1.0
Testing accuracy: 0.9


In [ ]:
# K-fold cross-validation

In [17]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

X, y = load_iris(return_X_y=True)
model = RandomForestClassifier(random_state=42)

scores = cross_val_score(
    model, X, y,
    cv=5,
    scoring="accuracy"
)

print("Scores for each fold:", scores)
print("Mean accuracy:", scores.mean())
print("Standard Variation:", scores.std())

Scores for each fold: [0.96666667 0.96666667 0.93333333 0.96666667 1.        ]
Mean accuracy: 0.9666666666666668
Standard Variation: 0.02108185106778919


In [20]:
# Data Leakage

In [ ]:
# Class Imbalance - Data from one category is more than other categories (Biased Output)

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
accuracy_score, precision_score, recall_score, f1_score)


# 95 normal cases (0) and 5 positive cases (1)
y_true = [0] * 95 + [1] * 5

# Baseline predicts the majority class every time
X = [[i] for i in range (len(y_true))]
model = DummyClassifier(strategy="most_frequent")
model.fit(X, y_true)
y_pred = model.predict(X)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1-score:", f1_score(y_true, y_pred))

Accuracy: 0.95
Precision: 0.0
Recall: 0.0
F1-score: 0.0


C:\Users\Laptop Solutions\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
# Remedies for Class Imbalance
# - Random Oversampling of Minority Class
# - Random Undersampling of Majority Class
# - SMOTE (create synthetic examples around the minority class)
# - Class Weights
# - Threshold Tuning (set threshold at 99.5% instead of 99%)
# - Collect more minority data or suitable augmentation

In [ ]:
# Implementing Remedies

In [14]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, average_precision_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE

# 1. Generate synthetic dataset with 95% zeros and 5% ones
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    weights=[0.95, 0.05],  # 95% class 0, 5% class 1
    random_state=42
)

# 2. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Define models dictionary
models = {
    "Baseline": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    ),
    "Class weights": make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=1000)
    ),
    "Random oversampling": ImbPipeline([
        ("scaler", StandardScaler()),
        ("sampling", RandomOverSampler(random_state=42)),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "SMOTE": ImbPipeline([
        ("scaler", StandardScaler()),
        ("sampling", SMOTE(random_state=42)),
        ("model", LogisticRegression(max_iter=1000))
    ])
}

# 4. Model Training & Evaluation Loop
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    print("\n---", name, "---")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Average Precision:", round(average_precision_score(y_test, y_score), 3))


--- Baseline ---
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       189
           1       0.00      0.00      0.00        11

    accuracy                           0.94       200
   macro avg       0.47      0.50      0.49       200
weighted avg       0.89      0.94      0.92       200

Average Precision: 0.337

--- Class weights ---
              precision    recall  f1-score   support

           0       0.98      0.70      0.82       189
           1       0.12      0.73      0.21        11

    accuracy                           0.70       200
   macro avg       0.55      0.72      0.52       200
weighted avg       0.93      0.70      0.79       200

Average Precision: 0.303

--- Random oversampling ---
              precision    recall  f1-score   support

           0       0.98      0.71      0.83       189
           1       0.13      0.73      0.22        11

    accuracy                           0.71       200
   macro

In [ ]:
# Classification Metrics

In [3]:
from sklearn.metrics import (
accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, balanced_accuracy_score
)

y_true = [1,1,1,1,0,0,0,0]
y_pred = [1,1,0,1,0,0,1,0]

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1-score:", f1_score(y_true, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("Accuracy:", classification_report(y_true, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_true, y_pred))

Accuracy: 0.75
Precision: 0.75
Recall: 0.75
F1-score: 0.75
Confusion Matrix:
 [[3 1]
 [1 3]]
Accuracy:               precision    recall  f1-score   support

           0       0.75      0.75      0.75         4
           1       0.75      0.75      0.75         4

    accuracy                           0.75         8
   macro avg       0.75      0.75      0.75         8
weighted avg       0.75      0.75      0.75         8

Balanced Accuracy: 0.75


In [ ]:
# Regression Evaluation Metrics

In [22]:
from sklearn.metrics import (
mean_absolute_error, mean_squared_error, r2_score
)

y_true = [100,200,300]
y_pred = [110,190,280]

mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_true, y_pred)

print("MAE:", mae)
print("MSE", mse)
print("RMSE", rmse)
print("R2", r2)

MAE: 13.333333333333334
MSE 200.0
RMSE 14.142135623730951
R2 0.97


In [ ]:
# SMOTE Method


--- Baseline ---
              precision    recall  f1-score   support

           0       0.90      0.98      0.94       179
           1       0.40      0.10      0.15        21

    accuracy                           0.89       200
   macro avg       0.65      0.54      0.55       200
weighted avg       0.85      0.89      0.86       200

Average Precision: 0.287

--- Class weights ---
              precision    recall  f1-score   support

           0       0.94      0.76      0.84       179
           1       0.22      0.57      0.32        21

    accuracy                           0.74       200
   macro avg       0.58      0.67      0.58       200
weighted avg       0.86      0.74      0.78       200

Average Precision: 0.279

--- Random oversampling ---
              precision    recall  f1-score   support

           0       0.93      0.75      0.83       179
           1       0.20      0.52      0.29        21

    accuracy                           0.73       200
   macro